In [ ]:
# urgent smoothing for one week-surgeon
def urgent_smooth(df_surgeon, df_surgeon_ori, working_days):

    # check if it's already smoothed
    sum_urgent = (df_surgeon['status'] != 0).sum()
    average = sum_urgent / len(working_days)
    continue_code = 0
    for day in working_days:
        urgent_cases = len(df_surgeon[(df_surgeon['weekday'] == day) & (df_surgeon['status'] != 0)])
        if abs(urgent_cases - average) >= 1:
            continue_code = 1
    # if it is already smoothed
    if continue_code == 0:
        return None
    
    obj_current = cal_obj1_urgent(df_surgeon_ori, working_days) # current value of objective 1

    iter_count = 0 # record the iteration count

    # all the movable cases in each day
    movable = {}
    for day in df_surgeon_ori['weekday'].unique():
        daily_schedule = df_surgeon_ori[df_surgeon_ori['weekday'] == day]
        movable[day] = daily_schedule[daily_schedule['status'] == 1].index.tolist() # index of status 1 cases under original schedule
    # add additional day to movable
    for day in working_days:
        if day not in df_surgeon_ori['weekday'].unique():
            movable[day] = []
    
    while True:
        # create a copy for restoration if objective value does not improve
        df_surgeon_copy = df_surgeon.copy()

        # for every day except the last day
        for day in working_days[:-1]:
            next_day = day + 1
            movable_cases = []
            # the day is an original working day
            if day in df_surgeon['weekday'].unique():
                schedule = df_surgeon[df_surgeon['weekday'] == day]
                total_urgent = (schedule['status'] != 0).sum() # number of urgent cases on the day
            # the day is an additional day
            else:
                total_urgent = 0

            # if next day is a working day
            if next_day in working_days:
                # the next day is an original working day
                if next_day in df_surgeon['weekday'].unique():
                    next_schedule = df_surgeon[df_surgeon['weekday'] == next_day]
                    next_total_urgent = (next_schedule['status'] != 0).sum()
                # the next day is an additional day
                else:
                    next_total_urgent = 0

                # check if these two days are smoothed
                if abs(total_urgent - next_total_urgent) >= 2:
                    # smooth these two days to average
                    target_next = round((total_urgent + next_total_urgent) / 2) # ideal number of cases next day
                    target = total_urgent + next_total_urgent - target_next # ideal number of cases this day

                    # move the case from this day to the next day
                    if target < total_urgent:
                        num_smooth = total_urgent - target # ideal number of cases to move to the next day
                        movable_cases = movable[day] # index of status 1 cases under original schedule on this day
                        actual_smooth = min(num_smooth, len(movable_cases)) # actual number of cases to move to the next day

                        for index in movable_cases[0:actual_smooth]:
                            df_surgeon.loc[index, 'weekday'] = next_day # move the case to the next day
                        
                        # update
                        del movable_cases[:actual_smooth] # the cases that are moved away cannot be moved in next iteration
                        movable[day] = movable_cases

                    # move the case from the next day to this day
                    elif target > total_urgent:
                        num_smooth = target - total_urgent # ideal number of cases to move to this day
                        movable_cases = movable[next_day] # index of status 1 cases under original schedule on this day
                        actual_smooth = min(num_smooth, len(movable)) # actual number of cases to move to this day

                        for index in movable_cases[0:actual_smooth]:
                            df_surgeon.loc[index, 'weekday'] = day # move the case to this day

                        # update
                        del movable_cases[:actual_smooth] # the cases that are moved away cannot be moved in next iteration
                        movable[next_day] = movable_cases

        obj_new = cal_obj1_urgent(df_surgeon, working_days) # calculate new objective 1 value
        if obj_new != 0:
            if obj_new < obj_current:
                obj_current = obj_new # enter next iteration if objective value improves
            else:
                # restore to previous iteration
                df_surgeon = df_surgeon_copy
                break
        
        # check the number of iterations
        if iter_count < 10:
            iter_count += 1
        else:
            break # maximum 10 iterations

    # check for monotonical increasing/decreasing special case
    if mono_check(df_surgeon, working_days) == None:
        # check if the objective 1 value improves
        if cal_obj1_urgent(df_surgeon, working_days) < cal_obj1_urgent(df_surgeon_ori, working_days):
            return df_surgeon # return new schedule if objective 1 value improves
        else:
            return None # return None if it is still the old schedule
    else:
        conse_diff, conse_mono_day = mono_check(df_surgeon, working_days)

    # monotonically increasing
    if conse_diff == -1:
        obj_new = 0
        # check from the second day to the last day
        for day in conse_mono_day[1:]:
            # if there is 1 urgent case that can be moved
            if len(movable[day]) > 0:
                schedule = df_surgeon[df_surgeon['weekday'] == day]
                total_urgent_case = (schedule['status'] != 0).sum()
                # for the last day
                if day == conse_mono_day[-1]:
                    total_urgent_case -= 1 # 1 less urgent case
                obj_new += total_urgent_case ** 2
            else:
                obj_new += 99999 # give it a large number to not move the case
                break
        
        for day in df_surgeon['weekday'].unique():
            # for those days that are not consecutive monotonically increasing days
            if day not in conse_mono_day:
                schedule = df_surgeon[df_surgeon['weekday'] == day]
                total_urgent_case = (schedule['status'] != 0).sum()
                obj_new += total_urgent_case ** 2
            # for the first consecutive monotonically increasing day
            elif day == conse_mono_day[0]:
                schedule = df_surgeon[df_surgeon['weekday'] == day]
                total_urgent_case = (schedule['status'] != 0).sum() + 1 # 1 more urgent case
                obj_new += total_urgent_case ** 2

        # if there is 1 movable urgent case from the second day to the last day,
        # and the objective value improves
        if obj_new < obj_current:
            for day in conse_mono_day[1:]:
                movable_days = movable[day]
                index = movable_days[0]
                df_surgeon.loc[index, 'weekday'] = (day - 1) # move 1 case to the previous day
        
    # monotonically decreasing
    elif conse_diff == 1:
        obj_new = 0
        # start from the first day to the second last day
        for day in conse_mono_day[:-1]:
            # if there is 1 urgent case that can be moved
            if len(movable[day]) > 0:
                schedule = df_surgeon[df_surgeon['weekday'] == day]
                total_urgent_case = (schedule['status'] != 0).sum()
                # for the first day
                if day == conse_mono_day[0]:
                    total_urgent_case -= 1 # 1 less urgent case
                obj_new += total_urgent_case ** 2
            else:
                obj_new += 99999 # give it a large number to stop
                break
        
        for day in df_surgeon['weekday'].unique():
            # for those days that are not consecutive monotonically decreasing days
            if day not in conse_mono_day:
                schedule = df_surgeon[df_surgeon['weekday'] == day]
                total_urgent_case = (schedule['status'] != 0).sum()
                obj_new += total_urgent_case ** 2
            # for the last consecutive monotonically decreasing day
            elif day == conse_mono_day[-1]:
                schedule = df_surgeon[df_surgeon['weekday'] == day]
                total_urgent_case = (schedule['status'] != 0).sum() + 1 # 1 more urgent case
                obj_new += total_urgent_case ** 2

        # if there is 1 movable urgent case from the first day to the second last day
        # and the objective value improves
        if obj_new < obj_current:
            for day in conse_mono_day[:-1]:
                movable_days = movable[day]
                index = movable_days[0]
                df_surgeon.loc[index, 'weekday'] = (day + 1) # move 1 case to the next day

    # check if the objective 1 value improves
    if cal_obj1_urgent(df_surgeon, working_days) < cal_obj1_urgent(df_surgeon_ori, working_days):
        return df_surgeon # return new schedule if objective 1 value improves
    else:
        return None # return None if it is still the old schedule

In [ ]:
# calculate objective 1 value for urgent cases in one week-surgeon
def cal_obj1_urgent(df_surgeon, working_days):
    obj1 = 0
    for day in working_days:
        total_urgent = len(df_surgeon[(df_surgeon['weekday'] == day) & (df_surgeon['status'] != 0)])
        obj1 += total_urgent ** 2
    return obj1

In [ ]:
# check for monotonically increasing/decreasing days
def mono_check(df_surgeon, working_days):
    conse_day = [] # create a list that stores the consecutive days

    # for every working day
    for day in working_days:
        conse_day_temp = []
        # if the next day exists
        while day + 1 in working_days:
            conse_day_temp.append(day) # add to the list
            day = day + 1
        # if there are at least 3 consecutive days
        if len(conse_day_temp) >= 2:
            conse_day_temp.append(day) # add the last day to the list
            conse_day = conse_day_temp
            break
    
    # return None if there are no consecutive days
    if not any(conse_day):
        return None
    
    # stores the difference in number of urgent cases with next day,
    # length is 1 less than the number of days
    diff_list = []

    for day in conse_day[:-1]:
        day_schedule = df_surgeon[df_surgeon['weekday'] == day] # schedule of this day
        next_day_schedule = df_surgeon[df_surgeon['weekday'] == (day + 1)] # schedule of the next day
        day_case = len(day_schedule[day_schedule['status'] != 0]) # total urgent cases on this day
        next_day_case = len(next_day_schedule[next_day_schedule['status'] != 0]) # total urgent cases on the next day

        diff = day_case - next_day_case
        diff_list.append(diff)
    
    day_index = [] # stores the index of the consecutive monotonically increasing/decreasing days
    conse_diff = 0 # indicator of monotonically increasing of decreasing

    # only consider at least 3 consecutive monotonically increasing/decreasing days
    for index in range(len(diff_list) - 1):
        if diff_list[index] in [1, -1]:
            diff = diff_list[index]
            # for the followings
            for index2 in range((index + 1),len(diff_list)):
                # 2 equal values indicate 3 consecutive monotonically increasing/decreasing days
                if diff_list[index2] in [diff, 0]:
                    day_index.append(index2) # add the following consecutive monotonically increasing/decreasing day to list  
        if day_index:
            day_index.insert(0, day_index[0] - 1) # add the index of first day to list
            conse_diff = diff # update the indicator
            break # in our case, no 2 seperate consecutive monotonically increasing/decreasing series
    
    conse_mono_day = [] # stores the consecutive monotonically increasing/decreasing days

    for index in day_index:
        conse_mono_day.append(conse_day[index])
        if index == day_index[-1]:
            conse_mono_day.append(conse_day[index + 1]) # add the last day of series to list
    
    return conse_diff, conse_mono_day

In [ ]:
# elective smoothing for one week-surgeon
def elective_smooth(df_surgeon, working_days):
    # save the original schedule
    df_surgeon_ori = df_surgeon.copy()

    obj1_ori = cal_obj1(df_surgeon_ori) # calculate the original objective 1 value
    obj2_ori = cal_obj2(df_surgeon_ori) # calculate the original objective 2 value

    elective_index = df_surgeon[df_surgeon['status'] == 0].index.tolist() # index of all elective cases
    
    df_surgeon.loc[df_surgeon['status'] == 0, 'weekday'] = 0 # set the weekday of elective cases to 0
    obj1_dict = {} # store the objective 1 value for each working day

    for day in working_days:
        if day in df_surgeon['weekday'].unique():
            obj1 = cal_obj1(df_surgeon[df_surgeon['weekday'] == day]) # calculate the objective 1 value for each original working day
            obj1_dict[day] = obj1
        else:
            obj1_dict[day] = 0 # set objective 1 value to 0 for the additional working day

    # move every elective case to the day with the smallest objective value
    for index in elective_index:
        min_obj = min(obj1_dict.values())
        target_day_list = [key for key, value in obj1_dict.items() if value == min_obj] # find the days with the smallest objective 1 value
        
        # if the objective value ties
        if len(target_day_list) > 1:
            # move to the day with the smallest objective 2 value
            obj2_dict = {}

            # find the smallest objective 2 value in these days
            for day in target_day_list:
                if day in df_surgeon['weekday'].unique():
                    obj2 = cal_obj2(df_surgeon[df_surgeon['weekday'] == day]) # calculate the objective 2 value for each original working day
                    obj2_dict[day] = obj2
                else:
                    obj2_dict[day] = 0 # set objective 2 value to 0 for the additional working day

            min_obj2 = min(obj2_dict.values())
            target_day = [key for key, value in obj2_dict.items() if value == min_obj2][0] # select the day with the smallest objective 2 value
            df_surgeon.loc[index, 'weekday'] = target_day # move the elective case to the target day
            # update objective value after moving the case
            new_obj1 = cal_obj1(df_surgeon[df_surgeon['weekday'] == target_day])
            obj1_dict[target_day] = new_obj1
            new_obj2 = cal_obj2(df_surgeon[df_surgeon['weekday'] == target_day])
            obj2_dict[target_day] = new_obj2

        # if there is only 1 day with the smallest objective 1 value
        else:
            target_day = target_day_list[0]
            df_surgeon.loc[index, 'weekday'] = target_day # move the elective case to the target day
            # update objective value after moving the case
            new_obj1 = cal_obj1(df_surgeon[df_surgeon['weekday'] == target_day])
            obj1_dict[target_day] = new_obj1

    obj1_new = cal_obj1(df_surgeon) # calculate the new objective 1 value
    obj2_new = cal_obj2(df_surgeon) # calculate the new objective 2 value

    # if it does not improve,
    # meaning that it is useless recombination
    if obj1_new >= obj1_ori:
        if obj2_new >= obj2_ori:
            df_surgeon = df_surgeon_ori # restore to the original schedule

    if df_surgeon.equals(df_surgeon_ori):
        return None # return None if it is still the old schedule
    else:
        return df_surgeon

In [ ]:
# calculate objective 1 value in one week-surgeon
def cal_obj1(df_surgeon):
    obj1 = 0
    for day in df_surgeon['weekday'].unique():
        obj1 += len(df_surgeon[df_surgeon['weekday'] == day]) ** 2
    return obj1

In [ ]:
# calculate objective 2 value in one week-surgeon
def cal_obj2(df_surgeon):
    obj2 = 0
    for day in df_surgeon['weekday'].unique():
        total_ne = len(df_surgeon[(df_surgeon['weekday'] == day) & (df_surgeon['status'] != 0)])
        obj2 += total_ne * len(df_surgeon[df_surgeon['weekday'] == day])
    return obj2

In [ ]:
def additional_day(df_surgeon, df_surgeon_ori):
    working_days_ori = list(df_surgeon['weekday'].unique()) # the original working days
    add_days = [i for i in [1,2,3,4,5] if i not in working_days_ori] # non-working days that can be selected as an additional day
    
    # >= 3 cases, 1 working day,
    if len(working_days_ori) == 1 and len(df_surgeon) >= 3:
        # need to add 2 additional working days
        import itertools

        add_days_comb = list(itertools.combinations(add_days, 2)) # all combinations of 2 additional days
        working_days_comb = [] # all combinations of new working days

        for comb in add_days_comb:
            new_working_days = working_days_ori + list(comb)
            new_working_days.sort()
            new_working_days = [int(x) for x in new_working_days]
            working_days_comb.append(new_working_days)
    
    else:
        # add 1 additional working day
        working_days_comb = [] # all combinations of new working days

        for add_day in add_days:
            new_working_days = working_days_ori + [add_day]
            new_working_days.sort()
            new_working_days = [int(x) for x in new_working_days]
            working_days_comb.append(new_working_days)

    best_obj1 = 9999999
    best_obj2 = 9999999
    best_df_surgeon = None

    df_surgeon_old = df_surgeon.copy()

    # try every combination
    for working_days in working_days_comb:
        df_surgeon = df_surgeon_old.copy()
        # urgent smoothing
        df_surgeon_new = urgent_smooth(df_surgeon, df_surgeon_ori, working_days)
        # if it performs urgent smoothing
        if df_surgeon_new is not None:
            df_surgeon = df_surgeon_new
        
        # elective smoothing
        df_surgeon_new2 = elective_smooth(df_surgeon, working_days)
        # if it performs urgent smoothing
        if df_surgeon_new2 is not None:
            df_surgeon = df_surgeon_new2
        
        # check obj1 & obj 2
        obj1 = cal_obj1(df_surgeon)
        obj2 = cal_obj2(df_surgeon)

        # update
        if obj1 < best_obj1:
            best_obj1 = obj1
            best_df_surgeon = df_surgeon
        elif obj1 == best_obj1:
            if obj2 < best_obj2:
                best_obj2 = obj2
                best_df_surgeon = df_surgeon
            else:
                continue
        else:
            continue
    
    return best_df_surgeon

In [ ]:
def schedule_es(row, data, all_surgeons):
    import random
    week = row['weeknum'] # week of the case
    weekday = row['weekday'] # weekday of the case

    working_surgeon = list(data[(data['weeknum'] == week) & (data['weekday'] == weekday)]['surgeon'].unique()) # surgeons that are working on that day

    # 65% chance assign to a surgeon who is not working on that day
    assignment = 1 if random.random() >= 0.65 else 0

    # assign to a surgeon who is not working on that day
    if assignment == 1:
        idle_surgeon = [item for item in all_surgeons if item not in working_surgeon]
        # if there is no idle surgeon, return the row
        if len(idle_surgeon) == 0:
            return row
        else:
            target_surgeon = random.choice(idle_surgeon) # randomly assign to a surgeon that is not working
            row['surgeon'] = target_surgeon

    # assign to a surgeon who is working on that day
    else:
        # if it is the only case in the day
        if len(working_surgeon) == 0:
            return row

        # if it is not the only case in the case
        else:
            target_surgeon = random.choice(working_surgeon) # randomly assign to a surgeon that is working

            # # assign to the surgeon with the least number of cases on that day
            # surgeon_cases = data[(data['weeknum'] == week) & (data['weekday'] == weekday) & (data['surgeon'].isin(working_surgeon))]
            # surgeon_cases_count = surgeon_cases['surgeon'].value_counts()
            # target_surgeon = surgeon_cases_count.idxmin()
            # # if there is a tie, randomly select one
            # if (surgeon_cases_count == surgeon_cases_count.min()).sum() > 1:
            #     target_surgeon = random.choice(surgeon_cases_count[surgeon_cases_count == surgeon_cases_count.min()].index.tolist())
            
            row['surgeon'] = target_surgeon
    
    return row

In [ ]:
def objective_cal(df_surgeon):
    gamma_avg_1 = 0.455
    gamma_avg_2 = 1.109
    gamma_avg_3 = 1.166
    gamma_el_1 = 0.567
    gamma_el_2 = 0
    gamma_el_3 = 0
    gamma_ne_1 = 0.377
    gamma_ne_2 = 1.605
    gamma_ne_3 = 1.647
    
    obj_value_avg_1 = 0
    obj_value_avg_2 = 0
    obj_value_avg_3 = 0
    obj_value_het_1 = 0
    obj_value_het_2 = 0
    obj_value_het_3 = 0
    
    ne = 0

    for day in df_surgeon['weekday'].unique():
        n_st = len(df_surgeon[df_surgeon['weekday'] == day])
        n_st_el = len(df_surgeon[(df_surgeon['weekday'] == day) & (df_surgeon['status'] == 0)])
        n_st_ne = n_st - n_st_el
        ne += n_st_ne

        obj_value_avg_1 += n_st * (gamma_avg_1 * n_st_el + gamma_avg_1 * n_st_ne)
        obj_value_avg_2 += n_st * (gamma_avg_2 * n_st_el + gamma_avg_2 * n_st_ne)
        obj_value_avg_3 += n_st * (gamma_avg_3 * n_st_el + gamma_avg_3 * n_st_ne)
        obj_value_het_1 += n_st * (gamma_el_1 * n_st_el + gamma_ne_1 * n_st_ne)
        obj_value_het_2 += n_st * (gamma_el_2 * n_st_el + gamma_ne_2 * n_st_ne)
        obj_value_het_3 += n_st * (gamma_el_3 * n_st_el + gamma_ne_3 * n_st_ne)

    obj_value_avg_1 = round(obj_value_avg_1, 3)
    obj_value_avg_2 = round(obj_value_avg_2, 3)
    obj_value_avg_3 = round(obj_value_avg_3, 3)
    obj_value_het_1 = round(obj_value_het_1, 3)
    obj_value_het_2 = round(obj_value_het_2, 3)
    obj_value_het_3 = round(obj_value_het_3, 3)

    return obj_value_avg_1, obj_value_avg_2, obj_value_avg_3, obj_value_het_1, obj_value_het_2, obj_value_het_3

In [ ]:
def worker(data_ori, obj_value_df, obj_value_better_df, avg_work_day_sim, improved_weeks_sim, select_frac):
    from tqdm import tqdm
    import pandas as pd
    pd.options.mode.chained_assignment = None
    import numpy as np
    for i in tqdm(range(20)):
        # data_ori = pd.read_excel("ICU_avg_effect.xlsx", sheet_name = "Old Schedule")
        data_ori = data_ori.iloc[:, :5]

        # do not consider Saturday & Sunday
        data = data_ori[~data_ori['weekday'].isin([6, 7])]

        # filter out emergent & salvage cases
        data_es = data[data['status'].isin([2, 3])]

        # schedule without emergent/salvage cases
        data = data[data['status'].isin([0, 1])]

        # randomly select out a percentage of the urgent cases
        urgent_indices = data[data['status'] == 1].sample(frac = select_frac).index
        data_urgent = data.loc[urgent_indices]
        data = data.drop(urgent_indices)

        data_copy = data.copy()

        # urgent smoothing
        # for every week
        for week in data['weeknum'].unique():
            df = data[data['weeknum'] == week] # schedule of this week
            df_ori = data_ori[data_ori['weeknum'] == week] # original schedule of this week
            # for every surgeon
            for surgeon in df['surgeon'].unique():
                df_surgeon = df[df['surgeon'] == surgeon] # schedule of this week-surgeon
                df_surgeon_ori = df_ori[df_ori['surgeon'] == surgeon] # original schedule of this week-surgeon
                df_surgeon_new = urgent_smooth(df_surgeon, df_surgeon_ori, list(df_surgeon['weekday'].unique()))
                # if it performs urgent smoothing
                if df_surgeon_new is not None:
                    # urgent_smooth_cases.append(df_surgeon_new)
                    data[(data['weeknum'] == week) & (data['surgeon'] == surgeon)] = df_surgeon_new    
        
        # elective smoothing
        # for every week
        for week in data['weeknum'].unique():
            df = data[data['weeknum'] == week] # schedule of this week
            # for every surgeon
            for surgeon in df['surgeon'].unique():
                df_surgeon = df[df['surgeon'] == surgeon] # schedule of this week-surgeon
                df_surgeon_new = elective_smooth(df_surgeon, list(df_surgeon['weekday'].unique()))
                # if it performs urgent smoothing
                if df_surgeon_new is not None:
                    # elective_smooth_cases.append(df_surgeon_new)
                    data[(data['weeknum'] == week) & (data['surgeon'] == surgeon)] = df_surgeon_new
        
        # additional day
        additional_day_week_surgeon = []
        # for every week
        for week in data['weeknum'].unique():
            df = data[data['weeknum'] == week]

            # for every surgeon
            for surgeon in df['surgeon'].unique():
                df_surgeon = df[df['surgeon'] == surgeon]

                working_days = len(df_surgeon['weekday'].unique())
                
                # >= 4 cases,
                # < 3 working days
                if (len(df_surgeon) >= 4 and working_days < 3):
                    additional_day_week_surgeon.append((week, surgeon))
                # 3 cases,
                # 1 or 2 working day
                elif (len(df_surgeon) == 3 and working_days <= 2):
                    additional_day_week_surgeon.append((week, surgeon))
                # 2 cases,
                # 1 working day
                elif (len(df_surgeon) == 2 and working_days == 1):
                    additional_day_week_surgeon.append((week, surgeon))

        # additional day smoothing
        for week, surgeon in additional_day_week_surgeon:
            df_surgeon = data[(data['weeknum'] == week) & (data['surgeon'] == surgeon)]
            df_surgeon_ori = data_ori[(data_ori['weeknum'] == week) & (data_ori['surgeon'] == surgeon)]

            df_surgeon_new = additional_day(df_surgeon, df_surgeon_ori)

            # update
            # additional_day_smooth_cases.append(df_surgeon_new)
            data[(data['weeknum'] == week) & (data['surgeon'] == surgeon)] = df_surgeon_new
        
        data_es_urgent = pd.concat([data_es, data_urgent], ignore_index = True) # merge the urgent cases with the emergent/salvage cases

        # scheduling for emergent/salvage/urgent cases
        all_surgeons = list(data['surgeon'].unique())
        for index in data_es_urgent.index:
            row = data_es_urgent.loc[index]
            new_row = schedule_es(row, data, all_surgeons) # schedule the emergency/salvage case
            data_es_urgent.loc[index] = new_row # update the new schedule

        # add the emergent/salvage cases to the schedule
        data = pd.concat([data, data_es_urgent], axis = 0)
        data = data.sort_values(by = ['weeknum', 'index'], ascending = [True, True])
        # data = data.sort_index()

        # add Saturdays & Sundays back to the schedule
        Sat = data_ori[data_ori['weekday'] == 6]
        data = pd.concat([data, Sat])
        Sun = data_ori[data_ori['weekday'] == 7]
        data = pd.concat([data, Sun])
        data = data.sort_values(by = ['weeknum', 'index'], ascending = [True, True])
        data = data.reset_index(drop = True)

        # # calculate objective value
        # obj_value_avg_1 = 0
        # obj_value_avg_2 = 0
        # obj_value_avg_3 = 0
        # obj_value_het_1 = 0
        # obj_value_het_2 = 0
        # obj_value_het_3 = 0

        # for week in data['weeknum'].unique():
        #     df_week = data[data['weeknum'] == week]
        #     for surgeon in df_week['surgeon'].unique():
        #         df_surgeon = df_week[df_week['surgeon'] == surgeon]
        #         _1, _2, _3, _4, _5, _6 = objective_cal(df_surgeon)
                
        #         obj_value_avg_1 += _1
        #         obj_value_avg_2 += _2
        #         obj_value_avg_3 += _3
        #         obj_value_het_1 += _4
        #         obj_value_het_2 += _5
        #         obj_value_het_3 += _6
            
        # row = pd.DataFrame({'obj_value_avg_1': obj_value_avg_1, 'obj_value_het_1': obj_value_het_1, 'obj_value_avg_2': obj_value_avg_2,
        # 'obj_value_het_2': obj_value_het_2, 'obj_value_avg_3': obj_value_avg_3, 'obj_value_het_3': obj_value_het_3}, index = [0])
        # obj_value_df = pd.concat([obj_value_df, row]).reset_index(drop = True)

        # calculate objective value
        obj_value_avg_1 = 0
        obj_value_avg_2 = 0
        obj_value_avg_3 = 0
        obj_value_het_1 = 0
        obj_value_het_2 = 0
        obj_value_het_3 = 0

        # calculate average working days
        tot_work_days = 0
        count = 0

        # calculate improved weeks
        weekly_obj = []
        weekly_obj_ori = []

        for week in data['weeknum'].unique():
            # objective value
            df_week = data[data['weeknum'] == week]
            df_week_ori = data_ori[data_ori['weeknum'] == week]

            temp_obj_value_avg_1 = 0
            temp_obj_value_avg_2 = 0
            temp_obj_value_avg_3 = 0
            temp_obj_value_het_1 = 0
            temp_obj_value_het_2 = 0
            temp_obj_value_het_3 = 0

            temp_obj_value_avg_1_ori = 0
            temp_obj_value_avg_2_ori = 0
            temp_obj_value_avg_3_ori = 0
            temp_obj_value_het_1_ori = 0
            temp_obj_value_het_2_ori = 0
            temp_obj_value_het_3_ori = 0

            for surgeon in df_week['surgeon'].unique():
                df_surgeon = df_week[df_week['surgeon'] == surgeon]
                _1, _2, _3, _4, _5, _6 = objective_cal(df_surgeon)
                
                temp_obj_value_avg_1 += _1
                temp_obj_value_avg_2 += _2
                temp_obj_value_avg_3 += _3
                temp_obj_value_het_1 += _4
                temp_obj_value_het_2 += _5
                temp_obj_value_het_3 += _6

            for surgeon in df_week_ori['surgeon'].unique():
                df_surgeon = df_week_ori[df_week_ori['surgeon'] == surgeon]
                _1, _2, _3, _4, _5, _6 = objective_cal(df_surgeon)
                
                temp_obj_value_avg_1_ori += _1
                temp_obj_value_avg_2_ori += _2
                temp_obj_value_avg_3_ori += _3
                temp_obj_value_het_1_ori += _4
                temp_obj_value_het_2_ori += _5
                temp_obj_value_het_3_ori += _6
            
            obj_value_avg_1 += min(temp_obj_value_avg_1, temp_obj_value_avg_1_ori)
            obj_value_avg_2 += min(temp_obj_value_avg_2, temp_obj_value_avg_2_ori)
            obj_value_avg_3 += min(temp_obj_value_avg_3, temp_obj_value_avg_3_ori)
            obj_value_het_1 += min(temp_obj_value_het_1, temp_obj_value_het_1_ori)
            obj_value_het_2 += min(temp_obj_value_het_2, temp_obj_value_het_2_ori)
            obj_value_het_3 += min(temp_obj_value_het_3, temp_obj_value_het_3_ori)

            # average working days
            for surgeon in data[data['weeknum'] == week]['surgeon'].unique():
                df_surgeon = data[(data['weeknum'] == week) & (data['surgeon'] == surgeon)]
                work_day = len(df_surgeon['weekday'].unique())
                tot_work_days += work_day
                count += 1

            # improved weeks
            df_week2 = data[data['weeknum'] == week]
            obj_value_avg_3 = 0
            for surgeon in df_week2['surgeon'].unique():
                df_surgeon = df_week2[df_week2['surgeon'] == surgeon]
                _1, _2, _3, _4, _5, _6 = objective_cal(df_surgeon)
                obj_value_avg_3 += _3
            weekly_obj.append(obj_value_avg_3)

            df_week3 = data_ori[data_ori['weeknum'] == week]
            obj_value_avg_4 = 0
            for surgeon in df_week3['surgeon'].unique():
                df_surgeon = df_week3[df_week3['surgeon'] == surgeon]
                _1, _2, _3, _4, _5, _6 = objective_cal(df_surgeon)
                obj_value_avg_4 += _3
            weekly_obj_ori.append(obj_value_avg_4)

        row2 = pd.DataFrame({'obj_value_avg_1': obj_value_avg_1, 'obj_value_het_1': obj_value_het_1, 'obj_value_avg_2': obj_value_avg_2,
        'obj_value_het_2': obj_value_het_2, 'obj_value_avg_3': obj_value_avg_3, 'obj_value_het_3': obj_value_het_3}, index = [0])
        obj_value_better_df = pd.concat([obj_value_better_df, row2]).reset_index(drop = True)

        avg_work_day = tot_work_days / count
        avg_work_day_sim.append(avg_work_day)

        weekly_obj = np.array(weekly_obj)
        weekly_obj_ori = np.array(weekly_obj_ori)

        improved_weeks = []

        for i in range(len(weekly_obj)):
            if weekly_obj_ori[i] - weekly_obj[i] >= 0.1:
                improved_weeks.append(i)

        improved_weeks_sim.append(len(improved_weeks))
    
    return obj_value_df, obj_value_better_df, avg_work_day_sim, improved_weeks_sim